# 26 · Load latency and overload

## The engineering question
How does the whole retrieval API behave under ordinary requests and a burst beyond its admitted capacity?

**Learning outcomes:** measure real loopback HTTP end-to-end latency; separate policy, encoding, Qdrant transport, and decoding; distinguish first model use from warm traffic; report successful latency and error rate together; keep native work bounded after response deadlines.

The predeclared local objective is successful warm serial p95 ≤1,000 ms with success fraction ≥0.99, using 100 requests. We then issue a separate 100-request burst with client concurrency eight against one server worker. The API has no waiting queue: excess admitted requests receive 429 and a Retry-After header. This is an overload probe, not a production arrival-rate model. Cache is disabled to measure retrieval rather than repeated-response lookup.

Index setup is timed separately. Models are released before the first HTTP request, so first-use includes model loading, but filesystem caches are not cleared. Each subsequent request includes native CPU embedding, representation-contract verification, Qdrant search, policy checks, and JSON response. Uvicorn binds only loopback on an ephemeral port and is shut down at the end.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("26", "O", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
import asyncio, json, time
import httpx
from uuid import uuid4
from vector_masterclass.data import documents, queries
from vector_masterclass.chunking import chunk_document
from vector_masterclass.store import VectorIndex
from vector_masterclass.policy import PolicyStore, seed_demo_policy
from vector_masterclass.service import SearchService, create_app
from vector_masterclass.local_server import local_server
from vector_masterclass.models import release_models

objectives = json.loads((root() / "configs/objectives.json").read_text())["load"]
index = VectorIndex("vmc_26_" + uuid4().hex[:8])
index.create()
policy = PolicyStore(lesson.path / "policy.sqlite")
docs = documents(tenant="public")
seed_demo_policy(policy, docs)
started = time.perf_counter()
for doc in docs:
    index.upsert(chunk_document(doc), document_id=doc.id, include_sparse=False)
setup_seconds = time.perf_counter() - started
release_models()
service = SearchService(index, policy, workers=1, timeout=20, cache_size=0)
rows = []
texts = [q.text for q in queries("development")]


async def requests(url):
    async with httpx.AsyncClient(
        base_url=url, timeout=30, headers={"Authorization": "Bearer demo-public-token"}
    ) as client:

        async def one(phase, i):
            started = time.perf_counter()
            response = await client.post("/search", json={"query": texts[i % len(texts)], "k": 5})
            elapsed = 1000 * (time.perf_counter() - started)
            stages = response.json().get("stage_seconds", {}) if response.status_code == 200 else {}
            rows.append(
                {
                    "phase": phase,
                    "request": i,
                    "status": response.status_code,
                    "milliseconds": elapsed,
                    "retry_after": response.headers.get("retry-after"),
                    **{k + "_ms": 1000 * v for k, v in stages.items()},
                }
            )

        await one("first-model-use", 0)
        for i in range(objectives["warm_serial_samples"]):
            await one("warm-serial", i)
        gate = asyncio.Semaphore(8)

        async def limited(i):
            async with gate:
                await one("burst-concurrency-8", i)

        await asyncio.gather(*(limited(i) for i in range(100)))


try:
    with local_server(create_app(service)) as url:
        await requests(url)
finally:
    service.close()
    policy.close()
    index.drop()
    index.close()
results = lesson.table(pd.DataFrame(rows))
good = results[results.status == 200]
summary = good.groupby("phase").milliseconds.agg(
    count="count", median="median", p95=lambda x: x.quantile(0.95), p99=lambda x: x.quantile(0.99)
)
display(summary)
display(pd.crosstab(results.phase, results.status))
display(
    good.groupby("phase")[
        ["policy_ms", "contract_check_ms", "embedding_ms", "qdrant_request_ms", "decode_ms"]
    ].median()
)
lesson.table(pd.DataFrame(list(service.events)), "server-events")
warm = results[results.phase == "warm-serial"]
warm_p95 = float(warm.loc[warm.status == 200, "milliseconds"].quantile(0.95))
warm_success = float((warm.status == 200).mean())
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for phase, group in good.groupby("phase"):
    x = np.sort(group.milliseconds)
    axes[0].step(x, np.arange(1, len(x) + 1) / len(x), where="post", label=f"{phase} (n={len(x)})")
axes[0].set(
    xlabel="Successful request milliseconds",
    ylabel="Empirical cumulative fraction",
    title="Successful latency must be paired with errors",
    xscale="log",
)
axes[0].legend(fontsize=7)
pd.crosstab(results.phase, results.status).plot.bar(stacked=True, ax=axes[1])
axes[1].set(ylabel="Requests", title="Load and rejection counts")
axes[1].tick_params(axis="x", rotation=20)
fig.tight_layout()
lesson.figure(fig)
plt.show()

## Interpretation and exercise
The serial workload is closed-loop: a slow response reduces offered load. The bounded burst keeps at most eight client requests active, so it also does not model arbitrary open-loop arrivals. Fast 429 responses must never be mixed into successful latency percentiles to make the service appear faster. A single first-use sample has no meaningful tail percentile; p99 from 100 successful requests is still a weak estimate.

Stage times are measured inside the service; their sum excludes some routing, scheduling, serialization, validation, and HTTP overhead. The enclosing retrieval time includes the contract-check network request. No query text or bearer token is retained in the server event ring. Production traces need privacy/redaction/retention policies and request-correlated storage/model spans.

**Exercise:** derive a safe retry budget for a 429 burst and explain why synchronized retries can amplify overload. Add an arrival-time schedule for an open-loop experiment and record queueing plus dropped requests. Predict what changes when batching raises throughput but adds waiting time.

Unit tests separately verify that a 504 response does not free its worker until native computation finishes. Timeouts are not cancellation. Production needs process isolation for hard cancellation, autoscaling/headroom, representative distributions, long runs, uncertainty estimates, and failure tests. This lesson measures a local retrieval API; it does not include generation latency, which is recorded in the live RAG lessons.

In [ ]:
manifest = lesson.finish(
    {
        "real_http_requests": len(results) == 201,
        "warm_success_objective": warm_success >= objectives["successful_warm_serial_fraction"],
        "warm_latency_objective": warm_p95 <= objectives["successful_warm_serial_p95_ms"],
        "overload_observed": bool((results.status == 429).any()),
        "retry_after_on_rejections": bool(
            results.loc[results.status == 429, "retry_after"].eq("1").all()
        ),
        "no_unexpected_status": set(results.status) <= {200, 429},
        "stage_timings_measured": bool(good.embedding_ms.notna().all()),
    },
    observations={
        "objectives": objectives,
        "setup_seconds": setup_seconds,
        "warm_p95_ms": warm_p95,
        "warm_success_fraction": warm_success,
        "cache_enabled": False,
        "workers": 1,
        "burst_client_concurrency": 8,
        "cold_filesystem_claim": False,
        "generation_included": False,
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.